## 🎯 What is this notebook about?
In **Step 4 and 5**, the champion **Gradient Boosting** forecasting model (18.55% MAPE) was developed and its predictions converted into dynamic Doctor and Nurse FTE staffing schedules.

However, machine learning models deployed in clinical production face **model drift**: patient scheduling behaviors evolve, clinic operating hours change, and seasonal infection patterns shift over time. Without structured governance, forecast accuracy degrades.

In **Step 6**, the production **Model Refresh & Governance Protocol** is formalized:
1. **Production Refresh Cadence**: Establishes the weekly scoring schedule and monthly retraining protocol.
2. **Automated Drift Detection**: Defines performance thresholds (MAPE > 18.0% over 3 consecutive weeks) that trigger retraining.
3. **Production Architecture**: Maps the end-to-end data pipeline from Electronic Health Records (EHR) to weekly staffing rotas.
4. **Handover Deliverables**: Catalogs all production models, datasets, and governance artifacts delivered in the Nexora Care Flow solution.


#### 1. Define Production Governance Parameters
This formalizes the operational monitoring parameters, retraining frequency, and drift alert thresholds into a standardized JSON configuration.

- Production machine learning systems require explicit Service Level Agreements (SLAs). Formally defining review intervals and performance boundaries ensures alignment between clinical leadership and IT data engineering regarding model maintenance.

In [6]:
# Production Refresh Rules & Parameters
refresh_policy = {
    "project": "Nexora Care Flow",
    "cadence": "Weekly Forecasts / Monthly Retraining",
    "drift_trigger_mape": "18.0%",
    "consecutive_weeks_threshold": 3,
    "champion_model": "GradientBoostingRegressor",
    "primary_metric": "MAPE (%)",
    "secondary_metric": "RMSE"
}

import json
print("Model Refresh Governance Policy")
print(json.dumps(refresh_policy, indent=2))

Model Refresh Governance Policy
{
  "project": "Nexora Care Flow",
  "cadence": "Weekly Forecasts / Monthly Retraining",
  "drift_trigger_mape": "18.0%",
  "consecutive_weeks_threshold": 3,
  "champion_model": "GradientBoostingRegressor",
  "primary_metric": "MAPE (%)",
  "secondary_metric": "RMSE"
}


**Governance Policy Summary**:
- **Scoring Cadence**: Every Sunday night at 23:00 for the upcoming 4 calendar weeks.
- **Retraining Cadence**: Monthly (on the 1st of every month) incorporating newly recorded appointment logs.
- **Drift Breach Trigger**: 3 consecutive weeks with a **MAPE exceeding 18.0%**.

#### 2. Retraining & Monitoring Logic
This implements and evaluates a prototype drift detection function (`check_model_drift`) that monitors rolling weekly forecast errors against the 18.0% MAPE operational threshold.

**Why use a 3-consecutive-week rule?**
- A single outlier week (such as extreme weather causing temporary clinic closures) does not indicate algorithmic failure. Requiring **3 consecutive weeks** above the 18.0% MAPE ceiling distinguishes temporary operational anomalies from systemic data drift.


In [7]:
# Drift Detection Logic Prototype
def check_model_drift(weekly_mapes):
    consecutive_breaches = sum(1 for error in weekly_mapes[-3:] if error > 18.0)
    if consecutive_breaches >= 3:
        return "ALERT: Model Drift Detected. Triggering Immediate Monthly Retraining Pipeline."
    return "Model Operating Within Acceptable Parameters."

# Test with sample metrics
sample_stable_errors = [12.5, 14.1, 13.8]
sample_drift_errors = [19.2, 21.0, 18.5]

print("Stable Metrics Check:", check_model_drift(sample_stable_errors))
print("Drift Metrics Check:", check_model_drift(sample_drift_errors))

Stable Metrics Check: Model Operating Within Acceptable Parameters.
Drift Metrics Check: ALERT: Model Drift Detected. Triggering Immediate Monthly Retraining Pipeline.


- The function validates that stable error series (`12.5%`, `14.1%`, `13.8%`) reflect normal operating conditions, while correctly triggering a retraining alert when sustained drift occurs (`19.2%`, `21.0%`, `18.5%`).

#### Summary


- Clean processed datasets are stored in `data/processed/`.
- Trained champion model artifacts are stored in `model/`.
- All Jupyter Notebooks (`01` through `06`) are pre-executed with rendered outputs and visualizations.
